# Model Training and Evaluation Pipeline

Train all 6 required classification models, evaluate each using 5 metrics, plot confusion matrices, and save all `.pkl` files to `models/`.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

# Resolve paths regardless of CWD
BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..'))
MODELS_DIR    = os.path.join(BASE_DIR, 'models')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed')
os.makedirs(MODELS_DIR, exist_ok=True)

# Load preprocessed data
X_train = joblib.load(os.path.join(PROCESSED_DIR, 'X_train.pkl'))
X_test  = joblib.load(os.path.join(PROCESSED_DIR, 'X_test.pkl'))
y_train = joblib.load(os.path.join(PROCESSED_DIR, 'y_train.pkl'))
y_test  = joblib.load(os.path.join(PROCESSED_DIR, 'y_test.pkl'))

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test:  {y_test.shape}")

## Helper — Evaluate and Save Model

Reusable function to compute all 5 metrics, plot the confusion matrix, and serialize the model.

In [ ]:
def evaluate_and_save(model, model_name, display_name):
    """Train, evaluate, plot confusion matrix, and save model."""
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    cm   = confusion_matrix(y_test, y_pred)
    
    print(f"{'='*45}")
    print(f"  {display_name}")
    print(f"{'='*45}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  ROC-AUC   : {auc:.4f}")
    
    # Confusion matrix plot
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    plt.title(f'{display_name} — Confusion Matrix')
    plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()
    
    # Save model
    save_path = os.path.join(MODELS_DIR, f'{model_name}.pkl')
    joblib.dump(model, save_path)
    print(f"  Saved → {save_path}\n")
    
    return dict(Model=model_name, Accuracy=round(acc,4), Precision=round(prec,4),
                Recall=round(rec,4), F1=round(f1,4), ROC_AUC=round(auc,4))

### Model 1: Logistic Regression
A linear model estimating class probability via a sigmoid function. Sets a solid interpretable baseline.

In [ ]:
from sklearn.linear_model import LogisticRegression
result_lr = evaluate_and_save(
    LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42),
    'logistic_regression', 'Logistic Regression'
)

### Model 2: Decision Tree
Recursively partitions feature space using Gini impurity or information gain to create interpretable split rules.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
result_dt = evaluate_and_save(
    DecisionTreeClassifier(random_state=42),
    'decision_tree', 'Decision Tree'
)

### Model 3: Random Forest
Bagging ensemble of decision trees. Aggregates predictions by majority vote to reduce variance and control overfitting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
result_rf = evaluate_and_save(
    RandomForestClassifier(n_estimators=100, random_state=42),
    'random_forest', 'Random Forest'
)

### Model 4: K-Nearest Neighbors
Lazy distance-based classifier. Assigns the majority class label among the K nearest training instances.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
result_knn = evaluate_and_save(
    KNeighborsClassifier(n_neighbors=5),
    'knn', 'K-Nearest Neighbors'
)

### Model 5: Support Vector Machine
Constructs an optimal hyperplane maximizing the margin between classes. RBF kernel maps inputs into higher dimensions.

In [ ]:
from sklearn.svm import SVC
result_svm = evaluate_and_save(
    SVC(kernel='rbf', probability=True, random_state=42),
    'svm', 'Support Vector Machine'
)

### Model 6: XGBoost
Sequential gradient boosting on decision trees. Minimizes residuals iteratively for high-accuracy tabular predictions.

In [ ]:
from xgboost import XGBClassifier
result_xgb = evaluate_and_save(
    XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
    'xgboost', 'XGBoost'
)

## Results Summary Table

Compile all metrics into a DataFrame sorted by F1 Score descending, save to CSV, and display.

In [ ]:
results_df = pd.DataFrame([
    result_lr, result_dt, result_rf, result_knn, result_svm, result_xgb
])
results_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
results_df = results_df.sort_values('F1', ascending=False).reset_index(drop=True)

csv_path = os.path.join(MODELS_DIR, 'results_summary.csv')
results_df.to_csv(csv_path, index=False)
print(f"Results saved → {csv_path}\n")
display(results_df)